In [ ]:
import re
from pathlib import Path

SA_JSON = Path("/content/jlens-credentials/drive-sa.json")
DRIVE_HELPER = Path("/content/jlens-credentials/colab_drive.py")
WHEEL_DIRECTORY = Path("/content/drive/MyDrive/data/jlens-reasoning/wheels")
REQUIREMENTS = WHEEL_DIRECTORY / "requirements-colab.txt"
COMMIT_FILE = WHEEL_DIRECTORY / "project-commit.txt"
DIRTY_FILE = WHEEL_DIRECTORY / "project-dirty.txt"

if SA_JSON.is_file():
    if not DRIVE_HELPER.is_file():
        raise RuntimeError(
            "Service-account credentials are present but colab_drive.py was not "
            "uploaded; use scripts/run_colab_notebook.sh for unattended CLI runs"
        )
    import runpy

    runpy.run_path(str(DRIVE_HELPER), run_name="__main__")
else:
    from google.colab import drive

    drive.mount("/content/drive")

if not COMMIT_FILE.is_file():
    raise RuntimeError(f"Missing project commit marker: {COMMIT_FILE}")
PROJECT_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip()
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Project commit marker is invalid")
if not DIRTY_FILE.is_file():
    raise RuntimeError(f"Missing project dirty marker: {DIRTY_FILE}")
dirty_value = DIRTY_FILE.read_text(encoding="utf-8").strip()
if dirty_value not in {"true", "false"}:
    raise RuntimeError("Project dirty marker is invalid")
PROJECT_WORKING_TREE_DIRTY = dirty_value == "true"

wheels = sorted(WHEEL_DIRECTORY.glob("jlens_reasoning-*.whl"))
if not REQUIREMENTS.is_file():
    raise RuntimeError(f"Missing locked requirements: {REQUIREMENTS}")
if len(wheels) != 1:
    raise RuntimeError(
        f"Expected exactly one project wheel in {WHEEL_DIRECTORY}, found {len(wheels)}"
    )

wheel = wheels[0]
print(f"Installing locked environment from {REQUIREMENTS}")
%pip install -qq --disable-pip-version-check --requirement {REQUIREMENTS}
print(f"Installing project wheel {wheel.name}")
%pip install -qq --disable-pip-version-check --force-reinstall --no-deps {wheel}
print("Colab project installation complete")

del (
    COMMIT_FILE,
    DIRTY_FILE,
    DRIVE_HELPER,
    REQUIREMENTS,
    SA_JSON,
    WHEEL_DIRECTORY,
    dirty_value,
    wheel,
    wheels,
)

# FLenQA: frozen probes × static J-Lens

Does the **correct True/False task label remain linearly decodable** when a long prompt fails, and how does its probe direction map downstream? This notebook reports measurements without assuming that propagation weakens. It does not establish retention of all task information or causal use of the decoded label.

Run `notebooks/flenqa_probe_assets.ipynb`, then `notebooks/flenqa_probe_eval.ipynb` with the current wheel first. The evaluation notebook saves small score tables; this notebook only runs new forward/backward passes for selected examples. All outputs go under the configured artifact root.

In [ ]:
import hashlib
import json

import jlens
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import load_from_disk
from tqdm.auto import tqdm

from experiments.flenqa_probe_jlens.analysis import (
    matched_prompt_pairs,
    validate_split,
)
from experiments.flenqa_probe_jlens.constants import PROBE_CONFIG
from experiments.jlens_readout_sanity.constants import LENS_PATH, MODEL_NAME, MODEL_PATH
from jlens_reasoning.benchmarks.flenqa.dataset import normalize_rows, prepare_prompts
from jlens_reasoning.benchmarks.flenqa.lens import deterministic_topk
from jlens_reasoning.environments.colab import initialize_colab
from jlens_reasoning.evaluation_utils import answer_token_variants
from jlens_reasoning.probe_jlens import (
    probe_sensitivities,
    static_probe_projection,
)
from jlens_reasoning.probing import (
    load_probe_checkpoint,
    probe_input_contract,
    token_margin,
    validate_probe_input_contract,
)

context = initialize_colab(enable_wandb=False, require_cuda=True)
ASSET_DIR = context.checkpoints_dir / "flenqa-probe-assets-chat-v2"
EVAL_DIR = context.runs_dir / "flenqa-probe-eval-chat-v2"
RESULT_DIR = context.runs_dir / "flenqa-probe-jlens-chat-v2"
MODEL_OUTPUT_PATH = context.runs_dir / "flenqa-full-run" / "model_outputs.parquet"
SHORT_CTX, LONG_CTX = 2000, 3000  # Existing comparison; both exceed fit lengths.
STRONG_PROBABILITY = 0.8  # Prespecified score threshold; not calibrated confidence.
PAIRS_PER_LABEL_GROUP, SEED, TOP_K = 3, 1729, 10

## 1. Load frozen probe and evaluation assets

Scores are bound to the checkpoint and answer file by SHA-256. The existing split and row identifiers are preserved. Generated answers, probe states, and the saved next-token margin use the **same direct chat template** and final wrapped token. Version 2 requires matching tokenizer contracts and per-prompt token/mask hashes; old raw-prompt probes and result tables must be regenerated.

In [ ]:
checkpoint = load_probe_checkpoint(
    ASSET_DIR / "probes.pt",
    metadata_path=ASSET_DIR / "metadata.json",
    model_name=MODEL_NAME,
)
metadata = checkpoint["metadata"]
split_path = context.checkpoints_dir / "flenqa-probe-assets" / "problem_split.json"
split = json.loads(split_path.read_text())
manifest = json.loads((EVAL_DIR / "manifest.json").read_text())
assert (
    checkpoint["format_version"]
    == metadata["format_version"]
    == manifest["format_version"]
    == 2
)
assert checkpoint["metadata"] == metadata and metadata["split"] == split
assert metadata["context_sizes"] == [250, 500]
assert metadata["model_name"] == manifest["model_name"] == MODEL_NAME
assert metadata["input_format"] == manifest["input_format"] == "chat_template_direct"
assert metadata["input_contract"] == manifest["input_contract"]
assert manifest["model_answer_input_format"] == "chat_template_direct"
for name, path in (
    ("probes", ASSET_DIR / "probes.pt"),
    ("model_outputs", MODEL_OUTPUT_PATH),
    ("probe_results", EVAL_DIR / "probe_results.parquet"),
    ("auroc", EVAL_DIR / "auroc.parquet"),
):
    with path.open("rb") as handle:
        assert (
            manifest[name + "_sha256"]
            == hashlib.file_digest(handle, "sha256").hexdigest()
        )

## 2. Load held-out prompt-layer scores

The evaluation notebook already aligned prompts and graded the saved chat answers. Here, check the split, complete prompt/layer coverage, and finite scores. The hashes above detect stale source assets.

In [ ]:
dataset = load_from_disk(context.datasets_dir / "flenqa")
rows = normalize_rows(
    dataset["eval"] if hasattr(dataset, "keys") else dataset, full=True
)
problem_to_split = validate_split(split, rows)
prompts = prepare_prompts(r for r in rows if problem_to_split[r.problem_id] == "test")
prompt_by_id = {p.prompt_id: p for p in prompts}
results = pd.read_parquet(EVAL_DIR / "probe_results.parquet")
auroc = pd.read_parquet(EVAL_DIR / "auroc.parquet")
assert not results.duplicated(["prompt_id", "layer"]).any()
assert set(results.prompt_id) == set(prompt_by_id)
num_layers, hidden_dim = metadata["num_layers"], metadata["hidden_dim"]
assert set(checkpoint["layers"]) == set(results.layer) == set(range(num_layers))
assert len(results) == len(prompts) * num_layers
assert results[["input_sha256", "n_input_tokens"]].notna().all().all()
assert (
    results.groupby("prompt_id")[["input_sha256", "n_input_tokens"]]
    .nunique()
    .eq(1)
    .all()
    .all()
)
assert (
    np.isfinite(
        results[["probe_score", "gold_margin", "gold_probability", "output_margin"]]
    )
    .all()
    .all()
)

## 3. Define the failure cohort

A strong failure has a parsed wrong chat answer, a correct probe, and gold-label sigmoid score ≥ 0.8. The next-token output margin uses the same input as the probe and generated answer. Keep it separate because a next-token diagnostic is not a graded generated answer.

In [ ]:
results["parsed_model_wrong"] = results.model_answer.notna() & ~results.model_correct
results["strong_failure"] = (
    results.parsed_model_wrong
    & results.probe_correct
    & results.gold_probability.ge(STRONG_PROBABILITY)
)
results["gold_output_margin"] = (2 * results.label - 1) * results.output_margin
results["strong_failure_margin_wrong"] = (
    results.strong_failure & results.gold_output_margin.lt(0)
)
headline_layer = min(
    range(num_layers),
    key=lambda layer: (
        metadata["probe_metrics"][str(layer)]["validation"]["log_loss"],
        layer,
    ),
)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print(
    {
        "test_problems": len(split["problems"]["test"]),
        "headline_layer_from_validation": headline_layer,
        "fit_lengths": metadata["context_sizes"],
        "matched_lengths": [SHORT_CTX, LONG_CTX],
    }
)

## 4. Probe performance and strong long failures by layer

A “strong” case has a parsed wrong chat answer, a correct frozen probe, and gold-label sigmoid score ≥ 0.8. This threshold is fixed before inspecting results; under length shift it is a score, not calibrated probability. Unparseable model answers are not counted as parsed wrong answers.

Accuracy/strength summaries weight each underlying problem equally. Counts refer to unique prompts; their variants are dependent. AUROC is the existing descriptive, unique-prompt metric. The final probe entry is post-final-normalization; earlier entries are block residuals.

In [ ]:
by_problem = results.groupby(["layer", "ctx_size", "problem_id"], as_index=False).agg(
    accuracy=("probe_correct", "mean"),
    gold_probability=("gold_probability", "mean"),
    gold_margin=("gold_margin", "mean"),
)
performance = (
    by_problem.groupby(["layer", "ctx_size"], as_index=False)
    .agg(
        probe_accuracy=("accuracy", "mean"),
        mean_gold_probability=("gold_probability", "mean"),
        mean_gold_margin=("gold_margin", "mean"),
    )
    .merge(auroc, on=["layer", "ctx_size"], validate="one_to_one")
)
long_results = results[results.ctx_size == LONG_CTX]
failure_table = long_results.groupby("layer", as_index=False).agg(
    prompts=("prompt_id", "size"),
    parsed_wrong=("parsed_model_wrong", "sum"),
    strong_failures=("strong_failure", "sum"),
    same_input_margin_disagrees=("strong_failure_margin_wrong", "sum"),
)
strong_problem_counts = (
    long_results[long_results.strong_failure].groupby("layer").problem_id.nunique()
)
failure_table["strong_failure_problems"] = (
    failure_table.layer.map(strong_problem_counts).fillna(0).astype(int)
)
failure_table["fraction_strong_given_wrong"] = (
    failure_table.strong_failures / failure_table.parsed_wrong.replace(0, np.nan)
)
display(
    performance[performance.ctx_size == LONG_CTX]
    .merge(failure_table, on="layer")
    .round(3)
)
examples = (
    long_results[long_results.strong_failure]
    .sort_values(
        ["layer", "gold_probability", "prompt_id"], ascending=[True, False, True]
    )
    .groupby("layer", sort=True)
    .head(2)
)
display(
    examples[
        [
            "layer",
            "problem_id",
            "task",
            "label",
            "gold_probability",
            "gold_margin",
            "gold_output_margin",
        ]
    ]
)

### Plot performance across context lengths

The failure fraction conditions on parsed wrong chat answers. A zero denominator remains missing.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ctx, group in performance.groupby("ctx_size"):
    axes[0].plot(group.layer, group.probe_accuracy, label=str(ctx))
    axes[1].plot(group.layer, group.mean_gold_probability, label=str(ctx))
axes[0].set(ylabel="Problem-weighted probe accuracy", ylim=(0, 1))
axes[1].set(ylabel="Mean gold-label sigmoid score", ylim=(0, 1))
axes[0].legend(title="Nominal length", fontsize=8)
axes[2].plot(failure_table.layer, failure_table.fraction_strong_given_wrong)
axes[2].set(ylabel=f"Strong / parsed wrong chat answers ({LONG_CTX})", ylim=(0, 1))
for ax in axes:
    ax.set_xlabel("Probe layer (zero-based)")
    ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(RESULT_DIR / "probe_performance.png", dpi=150)
plt.show()

In [ ]:
performance.to_parquet(RESULT_DIR / "probe_performance.parquet", index=False)
failure_table.to_parquet(RESULT_DIR / "failure_by_layer.parquet", index=False)
long_results[long_results.strong_failure].to_parquet(
    RESULT_DIR / "strong_failures.parquet", index=False
)
long_results.groupby(["layer", "task", "label"], as_index=False).agg(
    prompts=("prompt_id", "size"),
    parsed_wrong=("parsed_model_wrong", "sum"),
    strong_failures=("strong_failure", "sum"),
).to_parquet(RESULT_DIR / "failure_task_label.parquet", index=False)

## 5. Average/static propagation and linear output-token directions

Let \(d_l=w_l/\|w_l\|\), positive toward True. The stored map has **target coordinates in rows and source coordinates in columns**. The installed `JacobianLens.transport` uses row-vector batches \(h^\top\bar J_l^\top\). Thus
\[
v_l=\bar J_l d_l,\qquad a_{l,t}=u_t^\top v_l=(\bar J_l^\top u_t)^\top d_l.
\]
The right-hand side is the dot product with the repository's existing `jlens_vector`. A nonsymmetric synthetic test checks both forms.

The pinned fitter averages, over fitting prompts and valid source positions \(p\), a **sum over valid downstream target positions \(p'\)** of \(\partial h_T[p']/\partial h_l[p]\). It is not the last-token Jacobian of a particular FLenQA prompt. The minimal saved checkpoint does not record target-layer/position-estimator metadata; the default fitter targets the final block, but the external asset's exact fitting provenance needs confirmation.

The standard J-Lens readout is \(W_U\,N(\bar J_l h)\), where \(N\) is final normalization. Here \(W_U v_l\) intentionally reports **linear vocabulary-direction scores before final normalization**, not changes in normalized lens logits. Its local logit derivative would be \(W_U DN(\bar J_l h)\bar J_l d_l\). The repository defines token-pulled-back directions and coordinates in their span, but no separate semantic concept dictionary. Token strings are therefore reported as vocabulary tokens.

Static values are constant across prompts and lengths. Norm differences across layers cannot by themselves establish a long-context effect. The norm baseline is the exact RMS response to a uniformly random unit direction, \(\|\bar J_l\|_F/\sqrt{d}\).

In [ ]:
causal_lm = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, local_files_only=True
).to(context.device)
tokenizer = transformers.AutoTokenizer.from_pretrained(
    MODEL_PATH, local_files_only=True
)
causal_lm.eval()
input_contract = probe_input_contract(tokenizer, config=PROBE_CONFIG)
validate_probe_input_contract(metadata, input_contract)
validate_probe_input_contract(manifest, input_contract)
causal_lm.requires_grad_(False)
assert int(causal_lm.config.num_hidden_layers) == num_layers
assert int(causal_lm.config.hidden_size) == hidden_dim
# Do not let the adapter add BOS and change the frozen probe's input.
lens_model = jlens.from_hf(causal_lm, tokenizer, force_bos=False)
lens = jlens.JacobianLens.from_pretrained(LENS_PATH)
available_layers = sorted(
    (set(lens.source_layers) & set(checkpoint["layers"])) - {num_layers - 1}
)
assert available_layers, "No compatible pre-normalization probe/J-Lens layers"
unembedding = causal_lm.get_output_embeddings().weight.detach().float().cpu()
valid_token_ids = torch.tensor(sorted(set(tokenizer.get_vocab().values())))
assert valid_token_ids.min() >= 0 and valid_token_ids.max() < len(unembedding)
true_ids = tuple(i for i, _ in answer_token_variants(tokenizer, ("True",)))
false_ids = tuple(i for i, _ in answer_token_variants(tokenizer, ("False",)))
assert true_ids and false_ids and set(true_ids).isdisjoint(false_ids)
assert (
    list(true_ids) == manifest["true_ids"] and list(false_ids) == manifest["false_ids"]
)
answer_direction = unembedding[list(true_ids)].mean(0) - unembedding[
    list(false_ids)
].mean(0)
assert answer_direction.norm() > 0

### Project the frozen probe direction

`static_probe_projection` normalizes the weight, computes `jlens_jacobian @ probe_direction`, then `unembedding @ projected_probe_direction`. The final normalized probe is excluded from these block-output maps.

In [ ]:
static_rows, token_rows = [], []
for layer in available_layers:
    jlens_jacobian = lens.jacobians[layer].float().cpu()
    probe_weight = checkpoint["layers"][layer]["weight"]
    projected_probe_direction, token_scores = static_probe_projection(
        jlens_jacobian, probe_weight, unembedding
    )
    norm = float(projected_probe_direction.norm())
    random_rms = float(jlens_jacobian.norm() / np.sqrt(hidden_dim))
    static_rows.append(
        {
            "layer": layer,
            "propagation_norm": norm,
            "random_direction_rms": random_rms,
            "norm_over_random_rms": norm / random_rms if random_rms else np.nan,
            "true_false_alignment": float(
                projected_probe_direction
                @ answer_direction
                / (projected_probe_direction.norm() * answer_direction.norm())
            )
            if norm
            else np.nan,
            "linear_true_false_effect": float(
                token_scores[list(true_ids)].mean()
                - token_scores[list(false_ids)].mean()
            ),
        }
    )
    for sign, name in ((1, "positive"), (-1, "negative")):
        signed_scores = sign * token_scores[valid_token_ids]
        count = min(TOP_K, int((signed_scores > 0).sum()))
        for token in deterministic_topk(signed_scores, k=count):
            token_id = int(valid_token_ids[token.token_id])
            token_rows.append(
                {
                    "layer": layer,
                    "direction": name,
                    "rank": token.rank,
                    "token_id": token_id,
                    "raw_token": tokenizer.convert_ids_to_tokens(token_id),
                    "token": tokenizer.decode(
                        [token_id], clean_up_tokenization_spaces=False
                    ),
                    "score": float(token_scores[token_id]),
                }
            )
static = pd.DataFrame(static_rows)
tokens = pd.DataFrame(
    token_rows,
    columns=["layer", "direction", "rank", "token_id", "raw_token", "token", "score"],
)
comparison = performance[performance.ctx_size == LONG_CTX].merge(
    static, on="layer", how="left", validate="one_to_one"
)
display(comparison.round(3))
display(tokens[tokens["rank"] <= 3])
print(
    {
        "projected_layers": available_layers,
        "unprojected_probe_layers": sorted(
            set(checkpoint["layers"]) - set(available_layers)
        ),
        "lens_averaged_prompts": lens.n_prompts,
    }
)

### Compare static directions across layers

The random-direction RMS gives a scale baseline; these static quantities do not vary with prompt length.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(static.layer, static.propagation_norm, label="Probe direction")
axes[0].plot(
    static.layer, static.random_direction_rms, "--", label="Random unit direction RMS"
)
axes[0].set(ylabel="Static propagation norm")
axes[0].legend()
axes[1].plot(static.layer, static.true_false_alignment)
axes[1].axhline(0, color="black", lw=0.7)
axes[1].set(ylabel="Cosine to linear True-minus-False direction", ylim=(-1, 1))
for ax in axes:
    ax.set_xlabel("Layer")
    ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(RESULT_DIR / "static_propagation.png", dpi=150)
plt.show()

In [ ]:
static.to_parquet(RESULT_DIR / "static_propagation.parquet", index=False)
tokens.to_parquet(RESULT_DIR / "vocabulary_directions.parquet", index=False)
comparison.to_parquet(RESULT_DIR / "layer_comparison.parquet", index=False)

## 6. Selected prompt-specific output sensitivities

Pair the same problem, task, padding type, and dispersion at the two declared lengths. All provenance conditions are validated, including those merged by prompt deduplication. Select cases using the **validation-chosen probe layer**, requiring score ≥ 0.8 at both lengths, before computing gradients. Failure transitions have a correct short chat answer and parsed wrong long chat answer; stable controls have correct chat answers at both lengths. Sample at most three distinct problems per label/cohort, seed 1729. This is a small selected case study, not a population estimate.

On the **aligned direct-chat input**, define \(m=\operatorname{mean}z_{\mathrm{True}}-\operatorname{mean}z_{\mathrm{False}}\) and
\[
s_l(x)=\nabla_{h_l[-1]}m(x)^\top d_l.
\]
For gold sign \(g\), the derivative of gold margin \(gm\) along the gold probe direction \(gd_l\) is the same \(s_l\), since \(g^2=1\). This handles False labels without sign cancellation in aggregates. The derivative includes the actual downstream network and final normalization. It is **prompt-specific output sensitivity along the probe direction**, not a J-Lens coefficient. Parameter gradients are disabled; recording hooks only expose the graph and do not change activations.

In [ ]:
pairs = pd.DataFrame(
    matched_prompt_pairs(prompts, short_ctx=SHORT_CTX, long_ctx=LONG_CTX),
    columns=[
        "problem_id",
        "task",
        "padding_type",
        "dispersion",
        "label",
        "short_id",
        "long_id",
    ],
)
headline = results[results.layer == headline_layer].set_index("prompt_id")
pair_candidates = []
for pair in pairs.drop_duplicates(["short_id", "long_id"]).to_dict("records"):
    short, long = headline.loc[pair["short_id"]], headline.loc[pair["long_id"]]
    strong_both = (
        short.gold_probability >= STRONG_PROBABILITY
        and long.gold_probability >= STRONG_PROBABILITY
    )
    cohort = None
    if short.model_correct and long.parsed_model_wrong:
        cohort = "failure-transition"
    elif short.model_correct and long.model_correct:
        cohort = "stable-control"
    if strong_both and cohort:
        pair_candidates.append({**pair, "cohort": cohort})
candidates = pd.DataFrame(pair_candidates, columns=[*pairs.columns, "cohort"])
selected = candidates.sample(frac=1, random_state=SEED).drop_duplicates(
    ["cohort", "problem_id"]
)
selected = selected.groupby(["cohort", "label"], group_keys=False).head(
    PAIRS_PER_LABEL_GROUP
)
selected.to_parquet(RESULT_DIR / "selected_pairs.parquet", index=False)
display(
    selected[["problem_id", "task", "label", "cohort", "padding_type", "dispersion"]]
)
print(
    {
        "validated_condition_pairs": len(pairs),
        "candidate_endpoint_pairs": len(candidates),
        "selected_pairs": len(selected),
        "selection_layer": headline_layer,
    }
)
selected_ids = sorted(set(selected.short_id) | set(selected.long_id))

### Differentiate the chat-prompt output margin

The shared probe–J-Lens module starts the graph at the first block and removes its hooks on exit. Check its block outputs against the probe feature boundary, then compare recomputed probe scores with the saved export before interpreting derivatives.

In [ ]:
def prompt_sensitivity(prompt):
    saved = {
        layer: saved_scores.loc[(prompt.prompt_id, layer)]
        for layer in checkpoint["layers"]
    }
    sensitivities = probe_sensitivities(
        causal_lm,
        tokenizer,
        prompt.text,
        checkpoint["layers"],
        config=PROBE_CONFIG,
        blocks=lens_model.layers,
        objective=lambda logits: token_margin(logits, true_ids, false_ids),
        saved_records=saved,
    )
    gold_sign = 2 * int(prompt.label) - 1
    return [
        {
            "prompt_id": prompt.prompt_id,
            "problem_id": prompt.problem_id,
            "layer": result.layer,
            "label": int(prompt.label),
            "gold_probe_margin": gold_sign * result.probe_score,
            "gold_output_margin": gold_sign * result.output_margin,
            "sensitivity": result.sensitivity,
        }
        for result in sensitivities
    ]

In [ ]:
saved_scores = results.set_index(["prompt_id", "layer"])
sensitivity_rows = []
for prompt_id in tqdm(selected_ids, desc="Selected chat-prompt sensitivities"):
    sensitivity_rows.extend(prompt_sensitivity(prompt_by_id[prompt_id]))
sensitivity = pd.DataFrame(
    sensitivity_rows,
    columns=[
        "prompt_id",
        "problem_id",
        "layer",
        "label",
        "gold_probe_margin",
        "gold_output_margin",
        "sensitivity",
    ],
)
assert len(sensitivity) == len(selected_ids) * num_layers
sensitivity.to_parquet(RESULT_DIR / "prompt_sensitivity.parquet", index=False)

### Compare each selected problem at the two lengths

Keep one row per pair and layer. Report long-minus-short sensitivity separately by cohort and gold label; empty cohorts remain empty.

In [ ]:
paired_rows = []
if len(sensitivity):
    indexed = sensitivity.set_index(["prompt_id", "layer"])
    for pair in selected.to_dict("records"):
        for layer in range(num_layers):
            short = indexed.loc[(pair["short_id"], layer)]
            long = indexed.loc[(pair["long_id"], layer)]
            paired_rows.append(
                {
                    **pair,
                    "layer": layer,
                    "sensitivity_short": short.sensitivity,
                    "sensitivity_long": long.sensitivity,
                    "delta_sensitivity": long.sensitivity - short.sensitivity,
                    "gold_probe_margin_short": short.gold_probe_margin,
                    "gold_probe_margin_long": long.gold_probe_margin,
                    "gold_output_margin_long": long.gold_output_margin,
                }
            )
paired = pd.DataFrame(
    paired_rows,
    columns=[
        *selected.columns,
        "layer",
        "sensitivity_short",
        "sensitivity_long",
        "delta_sensitivity",
        "gold_probe_margin_short",
        "gold_probe_margin_long",
        "gold_output_margin_long",
    ],
)
paired.to_parquet(RESULT_DIR / "paired_sensitivity.parquet", index=False)

In [ ]:
sensitivity_summary = paired.groupby(["cohort", "label", "layer"], as_index=False).agg(
    n_problems=("problem_id", "nunique"),
    sensitivity_short=("sensitivity_short", "mean"),
    sensitivity_long=("sensitivity_long", "mean"),
    delta_sensitivity=("delta_sensitivity", "mean"),
)
sensitivity_summary.to_parquet(RESULT_DIR / "sensitivity_summary.parquet", index=False)
display(sensitivity_summary[sensitivity_summary.layer == headline_layer])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for (cohort, label), group in sensitivity_summary.groupby(["cohort", "label"]):
    axes[0].plot(group.layer, group.sensitivity_long, label=f"{cohort}, label={label}")
    axes[1].plot(group.layer, group.delta_sensitivity)
axes[0].set(ylabel="Long-prompt output sensitivity")
axes[1].set(ylabel=f"Sensitivity change: {LONG_CTX} − {SHORT_CTX}")
for ax in axes:
    ax.axhline(0, color="black", lw=0.7)
    ax.set_xlabel("Layer")
    ax.grid(alpha=0.2)
if len(sensitivity_summary):
    axes[0].legend(fontsize=7)
else:
    axes[0].text(
        0.1,
        0.5,
        "No pairs meet the prespecified selection",
        transform=axes[0].transAxes,
    )
fig.tight_layout()
fig.savefig(RESULT_DIR / "prompt_sensitivity.png", dpi=150)
plt.show()

## 7. Measured findings and next experiment

Read the tables together: label decodability, **average/static propagation**, and **prompt-specific chat-input sensitivity** are different measurements. Static maps have no context-length axis. A correct high-score probe on one example is not proof of a reliable decoder; use held-out accuracy/AUROC, label/task breakdowns, and independent problem counts.

Lower sensitivity in selected failures would support a local association between retained label decodability and weaker readout. Equal or higher sensitivity is also an informative result. Neither establishes a causal explanation. Token rankings are signed linear scores and do not by themselves identify semantic concepts.

For a later intervention experiment, first rerun the aligned pipeline and reproduce the measurements. Freeze a layer/direction and outcome using development examples, then test a norm-controlled perturbation at the final input position against identity and random-direction controls on fresh held-out problems. Choose the layer from the measured decodability/readout pattern; do not select it by intervention success on this test set. No intervention is run here.

In [ ]:
summary = {
    "headline_layer": headline_layer,
    "strong_score_threshold": STRONG_PROBABILITY,
    "long_context": LONG_CTX,
    "layers_with_strong_failures": failure_table.loc[
        failure_table.strong_failures.gt(0), "layer"
    ].tolist(),
    "strong_failure_problems_by_layer": failure_table.set_index(
        "layer"
    ).strong_failure_problems.to_dict(),
    "static_norm_range": [
        float(static.propagation_norm.min()),
        float(static.propagation_norm.max()),
    ],
    "static_alignment_range": [
        float(static.true_false_alignment.min()),
        float(static.true_false_alignment.max()),
    ],
    "selected_failure_problems": int(
        selected.loc[selected.cohort.eq("failure-transition"), "problem_id"].nunique()
    ),
    "selected_control_problems": int(
        selected.loc[selected.cohort.eq("stable-control"), "problem_id"].nunique()
    ),
    "sensitivity_scope": "direct-chat local derivatives; same-input generated outcomes; selected case studies",
    "causal_conclusion": "No causal conclusion; descriptive associations only.",
}
display(summary)
run_metadata = {
    **manifest,
    "analysis_project_commit": PROJECT_COMMIT,
    "selection_seed": SEED,
    "short_ctx": SHORT_CTX,
    "long_ctx": LONG_CTX,
    "strong_threshold": STRONG_PROBABILITY,
    "pairs_per_label_group": PAIRS_PER_LABEL_GROUP,
    "headline_layer": headline_layer,
    "available_layers": available_layers,
    "lens_path": LENS_PATH,
    "lens_n_prompts": lens.n_prompts,
    "lens_estimator_provenance": "Pinned fitter uses downstream-position sum; external asset target/reduction metadata not saved",
}
with Path(LENS_PATH).open("rb") as handle:
    run_metadata["lens_sha256"] = hashlib.file_digest(handle, "sha256").hexdigest()
for name, value in (("summary.json", summary), ("run_metadata.json", run_metadata)):
    (RESULT_DIR / name).write_text(json.dumps(value, indent=2, sort_keys=True) + "\n")
print(
    f"Saved figures, complete tables, selected pair IDs, and provenance to {RESULT_DIR}"
)